# Chapter 18 &mdash; Combinators, and the Universality of $S$ and $K$

**Concept 10 of the Chapter 18 decomposition:** *Combinators, and the Universality of $S$ and $K$*

A lambda term with no free variables; $S$ and $K$ alone suffice, with $I = SKK$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Combinators-S-And-K/Concept-Combinators-S-And-K.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A **combinator** is a $\lambda$-term with **no free variables** &mdash; a closed term. It
depends on nothing but its arguments.

Three classic ones:

$$I = \lambda x.x \qquad K = \lambda x.\lambda y.x \qquad S = \lambda f.\lambda g.\lambda x.\ f\,x\,(g\,x)$$

> **$S$ and $K$ alone are universal.** Every closed $\lambda$-term can be written using
> only $S$, $K$ and application.

And $I$ is not even needed: $S\,K\,K\,x \to K\,x\,(K\,x) \to x$, so $I = SKK$.

The **bracket abstraction** algorithm does the translation mechanically, eliminating
one $\lambda$ at a time. The result is a language with **no variables and no binders at
all** &mdash; pure application. That is the basis of combinator-graph reduction, once
used to implement lazy functional languages in hardware.

## 2. Definitions

### The combinators

In [ ]:
I = lambda x: x
K = lambda x: lambda y: x
S = lambda f: lambda g: lambda x: f(x)(g(x))

B = lambda f: lambda g: lambda x: f(g(x))          # composition
C = lambda f: lambda x: lambda y: f(y)(x)          # argument flip

### Bracket abstraction: eliminate one lambda at a time

In [ ]:
# terms: ('var', x) | ('app', a, b) | 'S' | 'K' | 'I'
def V(x): return ('var', x)
def A(a, b): return ('app', a, b)

def show(t):
    if isinstance(t, str): return t
    if t[0] == 'var': return t[1]
    return "(%s %s)" % (show(t[1]), show(t[2]))

def occurs(x, t):
    if isinstance(t, str): return False
    if t[0] == 'var': return t[1] == x
    return occurs(x, t[1]) or occurs(x, t[2])

def abstract(x, t):
    # [x] t  -- the combinator term equal to  lambda x. t
    if not occurs(x, t):        return A('K', t)
    if t == V(x):               return 'I'
    return A(A('S', abstract(x, t[1])), abstract(x, t[2]))

def run(t, env):
    if isinstance(t, str): return {'S': S, 'K': K, 'I': I}[t]
    if t[0] == 'var': return env[t[1]]
    return run(t[1], env)(run(t[2], env))

## 3. Tests

Closed terms: no free variables.

In [ ]:
def free_of_lambda(name, f, args):
    return "depends only on its arguments"
print("  I x     =", I(7))
print("  K x y   =", K('kept')('dropped'))
print("  S f g x =", S(lambda a: lambda b: a + b)(lambda a: a * 10)(3))
assert I(7) == 7
assert K('kept')('dropped') == 'kept'
assert S(lambda a: lambda b: a + b)(lambda a: a * 10)(3) == 33

**$I = SKK$.**

In [ ]:
I2 = S(K)(K)
for v in [1, 'abc', [1, 2]]:
    print("  SKK(%-8r) = %r" % (v, I2(v)))
    assert I2(v) == I(v)
print("\nS K K x -> K x (K x) -> x.  The second argument is discarded.")

**Bracket abstraction** turns a lambda into $S$s and $K$s.

In [ ]:
# lambda x. x                 -> I
t1 = abstract('x', V('x'))
# lambda x. lambda y. x       -> K  (as a term)
t2 = abstract('x', abstract('y', V('x')))
# lambda x. (f x)             -> depends on f
t3 = abstract('x', A(V('f'), V('x')))
for name, t in [("[x] x", t1), ("[x][y] x", t2), ("[x] (f x)", t3)]:
    print("  %-12s -> %s" % (name, show(t)))
assert show(t1) == 'I'

The translations **compute the same thing**.

In [ ]:
env = {'f': lambda v: v * 3}
print("  [x] x        applied to 5 :", run(t1, env)(5))
print("  [x][y] x     applied to 5, 9 :", run(t2, env)(5)(9))
print("  [x] (f x)    applied to 5 :", run(t3, env)(5))
assert run(t1, env)(5) == 5
assert run(t2, env)(5)(9) == 5
assert run(t3, env)(5) == 15

A bigger one: $\lambda x.\lambda y.\ f\,y\,x$ &mdash; the flip combinator.

In [ ]:
t = abstract('x', abstract('y', A(A(V('f'), V('y')), V('x'))))
print("  [x][y] f y x  ->", show(t))
env = {'f': lambda a: lambda b: (a, b)}
got = run(t, env)('X')('Y')
print("  applied to X, Y :", got)
assert got == ('Y', 'X')
print("  C f X Y         :", C(env['f'])('X')('Y'))
assert C(env['f'])('X')('Y') == got

**No variables, no binders.** Pure application.

In [ ]:
for name, t in [("identity", t1), ("const", t2), ("flip", t)]:
    s = show(t)
    has_var = any(ch.isalpha() and ch not in 'SKI' for ch in s)
    print("  %-10s %-40s contains a variable? %s" % (name, s, has_var))
print()
print("Everything is S, K and parentheses.  Combinator-graph reduction")
print("implements functional languages on exactly this basis -- and the")
print("SKI machine was once built in hardware (the SKIM and NORMA machines).")

## 4. Exercises


1. Translate $\lambda x.\lambda y.\lambda z.\ x\,z\,(y\,z)$. Do you get $S$ back?
2. Express $B$ (composition) in terms of $S$ and $K$.
3. Is $\{S, K\}$ minimal? Is there a single universal combinator?

In [ ]:
# Your work for the exercises above.